# Processing ISFS high-rate data

### Imports

In [1]:
%cd /glade/u/home/myasears/m2hats-in-gdex

/glade/u/home/myasears/m2hats-in-gdex


In [2]:
# Imports required to run analysis code
import xarray as xr
from pathlib import Path
import sys
import glob
import numpy as np
from itertools import groupby
import re
import dask.array as da

from m2hats_to_zarr import restructure_m2hats

### Spin up Dask cluster on Casper

In [3]:
# Imports required for dask
import dask 
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from dask.distributed import performance_report

lustre_scratch  = "/lustre/desc1/scratch/myasears"

# Create a PBS cluster object
cluster = PBSCluster(
    job_name = 'dask-eol-26',
    cores = 1,
    memory = '20GiB',
    processes = 1,
    local_directory = lustre_scratch+'/dask/spill/',
    log_directory = lustre_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=1:mem=8GB',
    queue = 'casper',
    walltime = '5:00:00',
    interface = 'ext'
)

# Create the client to load the Dashboard
client = Client(cluster)

# Scale number of workers for use case
cluster.scale(jobs=20)
client.wait_for_workers(20)

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36095 instead
  warnings.warn(


In [4]:
cluster

PBSCluster(2d947251, 'tcp://128.117.208.119:45243', workers=20, threads=20, memory=160.00 GiB)

### Read in all ISFS tiltcor_geo files and concatenate into one xarray dataset

Considerations: Aligning the time dimension to easily concatenate; beforehand, they were all organized by seconds from midnight and required base_time to be added to complete the hour. 
Performed this in parallel using 20 workers, then saved the concatenated files as a Zarr store to more easily begin next stage of processing. 

In [20]:
isfs_highrate_path = "/lustre/desc1/scratch/myasears/M2HATS/FDA_datasets/surface_highrate_geo"

A single file in this dataset (isfs_m2hats_qc_geo_hr_20230728_06.nc) had flipped dimensions. Un-flipping them for this project.

In [21]:
normal_path = f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_03.nc"
normal_dataset = xr.open_dataset(normal_path)
print(normal_dataset.dims)

errored_path = f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_06.nc"
errored_dataset = xr.open_dataset(errored_path)
print(errored_dataset.dims)

FrozenMappingWarningOnValuesAccess({'time': 3600, 'sample': 60, 'sample_50': 50, 'sample_30': 30, 'sample_20': 20})
FrozenMappingWarningOnValuesAccess({'time': 3600, 'sample': 50, 'sample_60': 60, 'sample_30': 30, 'sample_20': 20})


In [22]:
# Rename dims so they match the others:
#   sample (50)    -> sample_50
#   sample_60 (60) -> sample
# Use a temporary name to avoid a name collision during the swap.
errored_dataset = errored_dataset.rename({"sample": "__tmp_sample_50"})
errored_dataset = errored_dataset.rename({"sample_60": "sample"})
errored_dataset = errored_dataset.rename({"__tmp_sample_50": "sample_50"})

errored_dataset.to_netcdf(f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_06_aligned.nc")

In [23]:
# Locate and select files to be read in, replacing the errored file with the aligned version.
isfs_highrate_files = [
    f for f in sorted(Path(isfs_highrate_path).glob("*.nc"))
    if f.name != "isfs_m2hats_qc_geo_hr_20230728_06.nc"
]

Back to group processing.

In [24]:
_T1_RE = re.compile(r'^(u|v|w|tc|ldiag|diagbits|diag)_.+_t1$')

def fix_time(ds):
    """Override `time` with base_time + within-hour offset.

    M2HATS high-rate files store `time` as 'seconds within hour 0 of the
    file's day' rather than absolute, and `base_time` holds the hour offset.
    Defensive: if base_time is missing, or if `time` already exceeds the
    hour-0 range (meaning the file is already correctly decoded), pass
    through unchanged.
    """
    if "base_time" not in ds.variables:
        return ds
    if (ds["time"].dt.hour > 0).any():
        return ds.drop_vars("base_time", errors="ignore")
    base = ds["base_time"]
    tod = ds["time"] - ds["time"].dt.floor("D")
    return ds.assign_coords(time=base + tod).drop_vars("base_time")

def fix_time_and_t1(ds):
    ds = fix_time(ds)

    pre_swap = [v for v in ds.data_vars if _T1_RE.match(v) and 'sample_30' in ds[v].dims]
    post_swap = [v for v in ds.data_vars if _T1_RE.match(v) and 'sample' in ds[v].dims]

    if pre_swap:
        # Rename t1 -> t1p for the real data
        ds = ds.rename({v: re.sub(r'_t1$', '_t1p', v) for v in pre_swap})
        # Add NaN placeholder for post-swap t1 on 'sample' dim
        nt = ds.sizes['time']
        for v in pre_swap:
            ds[v] = xr.DataArray(
                da.full((nt, 60), np.nan, dtype='float32', chunks=(nt, 60)),
                dims=('time', 'sample')
            )

    if post_swap:
        # Add NaN placeholder for pre-swap t1p on 'sample_30' dim
        nt = ds.sizes['time']
        for v in post_swap:
            ds[re.sub(r'_t1$', '_t1p', v)] = xr.DataArray(
                da.full((nt, 30), np.nan, dtype='float32', chunks=(nt, 30)),
                dims=('time', 'sample_30')
            )

    return ds

In [ ]:
zarr_path = "/lustre/desc1/scratch/myasears/M2HATS/WORKING_datasets/isfs_m2hats_qc_geo_combined.zarr"


def date_key(p):
    m = re.search(r'_(\d{8})_', p.name)
    return m.group(1) if m else "unknown"

# Detect which dates are already written so we can resume
import zarr as zarr_lib
try:
    existing = xr.open_zarr(zarr_path)
    last_written = str(existing.time.values[-1])[:10].replace("-", "")
    print(f"Resuming after {last_written}")
    existing.close()
    first_run = False
except (FileNotFoundError, zarr_lib.errors.GroupNotFoundError):
    last_written = None
    first_run = True

for date, group in groupby(isfs_highrate_files, key=date_key):
    day_files = list(group)
    if last_written and date <= last_written:
        print(f"Skipping {date} (already written)")
        continue

    print(f"Processing {date} ({len(day_files)} files)…")
    ds_day = xr.open_mfdataset(
        day_files,
        preprocess=fix_time_and_t1,
        combine="nested",
        concat_dim="time",
        parallel=True,
        chunks={"time": 3600},
        decode_timedelta=False,
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )

    ds_day = ds_day.chunk({"time": 3600})   # normalize NaN placeholders and all other vars


    if first_run:
        ds_day.to_zarr(zarr_path, mode="w", consolidated=True)
        first_run = False
    else:
        ds_day.to_zarr(zarr_path, append_dim="time")

    ds_day.close()
    print(f"  Done {date}")


# Checkpoint for saving the dataset

In [5]:
ds = xr.open_zarr(
    "/lustre/desc1/scratch/myasears/M2HATS/WORKING_datasets/isfs_m2hats_qc_geo_combined.zarr",
)

In [6]:
ds

<xarray.Dataset> Size: 549GB
Dimensions:           (time: 5529600, sample: 60, sample_50: 50, sample_30: 30,
                       sample_20: 20)
Coordinates:
  * time              (time) datetime64[ns] 44MB 2023-07-23T00:00:00.500000 ....
Dimensions without coordinates: sample, sample_50, sample_30, sample_20
Data variables: (12/546)
    co2_0_5m_t0       (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    co2_15m_t0        (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    co2_1m_t0         (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    co2_28m_t0        (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    co2_2m_t0         (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    co2_3m_t0         (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    ...                ...
    w_4m_t5           (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    w_4m_t6           (time, sample_50) float32 1GB dask.array<chunksize=(3600, 50), meta=np.ndarray>
    w_4m_t7           (time, sample_50) float32 1GB dask.array<chunksize=(3600, 50), meta=np.ndarray>
    w_4m_t8           (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    w_4m_t9           (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
    w_7m_t0           (time, sample) float32 1GB dask.array<chunksize=(3600, 60), meta=np.ndarray>
Attributes:
    history:                   Created: 2024-02-29 20:32:47 +0000\n
    NIDAS_version:             v1.2.1-8
    calibration_file_path:     /net/isf/isff/projects/M2HATS/ISFS/cal_files/d...
    dataset:                   hr_qc_geo
    dataset_description:       High rate QC winds in geographic coordinates
    project_config:            /net/isf/isff/projects/M2HATS/ISFS/config/m2ha...
    wind3d_horiz_coordinates:  geographic
    file_length_seconds:       3600
    wind3d_horiz_rotation:     1
    wind3d_tilt_correction:    0

In [7]:
subset = ds.isel(time=slice(0, 300)).persist()    # one hour, in cluster memory

tree_test = restructure_m2hats(
    subset,
    output_path="./m2hats_test.zarr",
    tilt_corrected=True,
    time_chunk_seconds=300,
    write=False,                  # in-memory only for now
)

for path in tree_test.groups:
    node = tree_test[path].ds
    if not node.data_vars:
        continue
    print(f"{path:35s} dims={dict(node.sizes)}")

/array/sonic_60hz                   dims={'site': 22, 'time': 300, 'sample': 60}
/array/sonic_50hz                   dims={'site': 9, 'time': 300, 'sample_50': 50}
/array/sonic_30hz                   dims={'site': 20, 'time': 300, 'sample_30': 30}
/array/irga_60hz                    dims={'site': 16, 'time': 300, 'sample': 60}
/array/barometer_20hz               dims={'site': 16, 'time': 300, 'sample_20': 20}
/array/trh_1hz                      dims={'site': 16, 'time': 300}
/profile_t0/sonic_60hz              dims={'height': 8, 'time': 300, 'sample': 60}
/profile_t0/irga_60hz               dims={'height': 8, 'time': 300, 'sample': 60}
/profile_t0/barometer_20hz          dims={'height': 8, 'time': 300, 'sample_20': 20}
/profile_t0/trh_1hz                 dims={'height': 8, 'time': 300}


Expected output on a pre-Aug-8 hour (or any subset that doesn't span the t1 swap):

```
/array/sonic_60hz           dims={'site': 22, 'time': 3600, 'sample': 60}
/array/sonic_50hz           dims={'site': 9,  'time': 3600, 'sample_50': 50}
/array/sonic_30hz           dims={'site': 21, 'time': 3600, 'sample_30': 30}
/array/irga_60hz            dims={'site': 16, 'time': 3600, 'sample': 60}
/array/barometer_20hz       dims={'site': 16, 'time': 3600, 'sample_20': 20}
/array/trh_1hz              dims={'site': 16, 'time': 3600}
/profile_t0/sonic_60hz      dims={'height': 8, 'time': 3600, 'sample': 60}
/profile_t0/irga_60hz       dims={'height': 8, 'time': 3600, 'sample': 60}
/profile_t0/barometer_20hz  dims={'height': 8, 'time': 3600, 'sample_20': 20}
/profile_t0/trh_1hz         dims={'height': 8, 'time': 3600}
```

21 + 9 + 20 = 50 array towers, always. If you see an unexpected extra `sample` dim leaking into `sonic_30hz` or `sonic_50hz` it means your subset spans the t1 swap boundary — the next section handles that.

Spot-check a few real values to be sure the data isn't all NaN:

In [8]:
leaf = tree_test["/array/sonic_30hz"].ds
print(leaf.wind_u.isel(site=0, time=slice(0, 5)).compute())

<xarray.DataArray 'wind_u' (time: 5, sample_30: 30)> Size: 600B
array([[-6.6641917 , -6.6641917 , -5.6323314 , -3.7167919 , -4.0188847 ,
        -3.6289682 , -3.6082122 , -3.3830307 , -4.3194284 , -4.626797  ,
        -4.176567  , -3.1141033 , -1.7821094 , -1.9910337 , -1.9350607 ,
        -1.9712173 , -2.6298318 , -1.0936806 , -0.6276204 , -0.5857556 ,
        -0.91569024, -2.2482688 , -2.8245745 , -2.3503351 , -0.86754715,
        -3.301102  , -2.1080704 , -2.3845954 , -2.4451141 , -2.1624103 ],
       [-3.477981  , -3.1991785 , -2.9813814 , -2.9830647 , -3.6193948 ,
        -2.8530722 , -3.4180264 , -2.5491705 , -2.231176  , -1.5510178 ,
        -2.0029097 , -2.072686  , -2.127451  , -1.5083929 , -1.0932335 ,
        -2.0292053 , -2.4908757 , -2.1444757 , -2.1184454 , -2.0628939 ,
        -0.79429394, -1.9050074 , -2.511382  , -3.003193  , -3.901841  ,
        -3.6150665 , -3.8562298 , -3.9614089 , -3.7176473 , -3.4811108 ],
       [-2.8554852 , -3.4504132 , -3.5115838 , -3.2000153 

### Restructure the dataset
Change the format of the dataset from variable-based to have more coordinates. 

In [9]:
zarr_out = "/lustre/desc1/scratch/myasears/M2HATS/GDEX_datasets/isfs_m2hats_qc_geo.zarr"
tree = restructure_m2hats(ds, zarr_out, write=False, tilt_corrected=True)

In [11]:
import zarr as zarr_lib
import m2hats_to_zarr as m
import importlib
importlib.reload(m)
from m2hats_to_zarr import restructure_m2hats  # refresh the direct import too

TIME_CHUNK = 3600  # seconds

# Stamp the swap into root attrs so the store is self-documenting.
tree.attrs["t1_csat_swap_utc"] = "2023-08-23T13:00:00"
tree.attrs["t1_csat_swap_note"] = (
    "Site t1 sonic was a CSAT3 (30 Hz, site t1p) before this timestamp "
    "and a CSAT3A (60 Hz, site t1) after. Pre-swap data lives in "
    "/array/sonic_30hz under site t1p; post-swap data lives in "
    "/array/sonic_60hz under site t1. Each column is NaN-padded "
    "outside its valid period."
)

# Write root group (attrs only) first to initialise the store.
tree.ds.to_zarr(zarr_out, mode="w", consolidated=False)

# Write each leaf group separately — one Dask graph at a time.
for path in tree.groups:
    node = tree[path]
    if not node.ds.data_vars:
        continue
    enc = m._encoding_for(node.ds, TIME_CHUNK)
    # Align Dask chunks with Zarr encoding: non-time dims are written whole
    ds_w = node.ds.chunk({d: -1 for d in node.ds.dims if d != "time"})
    print(f"writing  {path:35s} ({ds_w.sizes.get('time', 0):>9,} steps)", flush=True)
    ds_w.to_zarr(zarr_out, group=path, mode="a", consolidated=False, encoding=enc)

zarr_lib.consolidate_metadata(zarr_out)
print("done.")

writing  /array/sonic_60hz                   (5,529,600 steps)


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=12, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/distributed/client.py:3374: UserWarning: Se

writing  /array/sonic_50hz                   (5,529,600 steps)


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/distributed/client.py:3374: UserWarning: Sen

writing  /array/sonic_30hz                   (5,529,600 steps)


...

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


writing  /profile_t0/trh_1hz                 (5,529,600 steps)


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/group.py:2711: ZarrUserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


done.


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=12, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/apps/opt/miniforge/envs/npl-2026a/

### Did it work?

In [14]:
tree = xr.open_datatree(
    zarr_out,
    engine="zarr",
    consolidated=True,
    chunks={},          # use stored chunk sizes → lazy Dask arrays
)


In [19]:
sonic = tree["/array/sonic_60hz"].ds
sonic


<xarray.DatasetView> Size: 175GB
Dimensions:                (site: 22, time: 5529600, sample: 60)
Coordinates:
  * site                   (site) object 176B 't0p' 't1' 't2' ... 't44' 't47'
  * time                   (time) datetime64[ns] 44MB 2023-07-23T00:00:00.500...
    csat_model_appendix_a  (site) <U12 1kB dask.array<chunksize=(22,), meta=np.ndarray>
    sample_offset          (sample) float64 480B dask.array<chunksize=(60,), meta=np.ndarray>
    site_height            (site) float64 176B dask.array<chunksize=(22,), meta=np.ndarray>
    site_lat               (site) float64 176B dask.array<chunksize=(22,), meta=np.ndarray>
    site_lon               (site) float64 176B dask.array<chunksize=(22,), meta=np.ndarray>
Dimensions without coordinates: sample
Data variables:
    sonic_diag_bits        (site, time, sample) float32 29GB dask.array<chunksize=(22, 3600, 60), meta=np.ndarray>
    sonic_qc_flag          (site, time, sample) float32 29GB dask.array<chunksize=(22, 3600, 60), meta=np.ndarray>
    sonic_temperature      (site, time, sample) float32 29GB dask.array<chunksize=(22, 3600, 60), meta=np.ndarray>
    valid                  (time) int8 6MB dask.array<chunksize=(3600,), meta=np.ndarray>
    wind_u                 (site, time, sample) float32 29GB dask.array<chunksize=(22, 3600, 60), meta=np.ndarray>
    wind_v                 (site, time, sample) float32 29GB dask.array<chunksize=(22, 3600, 60), meta=np.ndarray>
    wind_w                 (site, time, sample) float32 29GB dask.array<chunksize=(22, 3600, 60), meta=np.ndarray>
Attributes:
    subarray:          horizontal_50_tower
    instrument_class:  CSAT3A 3D sonic anemometer (60 Hz)
    Conventions:       CF-1.10

In [37]:
import pandas as pd
import numpy as np
import xarray as xr
from pathlib import Path
from m2hats_to_zarr import VAR_METADATA

# ── Parameters ────────────────────────────────────────────────────────────────
CHECK_TIME  = np.datetime64('2023-09-21T09:12:02.500000000')
CHECK_SITE  = "t0p"
CHECK_GROUP = "/array/sonic_60hz"   # t44 = CSAT3A+EC150 → sonic_60hz
# To test pre-swap t1:  CHECK_SITE="t1p", CHECK_GROUP="/array/sonic_30hz"
# To test post-swap t1: CHECK_SITE="t1",  CHECK_GROUP="/array/sonic_60hz"

ISFS_PATH = Path(isfs_highrate_path)

# ── 1. Zarr ───────────────────────────────────────────────────────────────────
# One time step × one site — tiny slice, no cluster needed
zarr_slice = (
    tree[CHECK_GROUP].ds
    .sel(time=CHECK_TIME, site=CHECK_SITE)
    .compute()
)

# ── 2. Raw NetCDF ─────────────────────────────────────────────────────────────
ts       = pd.Timestamp(CHECK_TIME.astype("datetime64[ms]").item())
raw_file = ISFS_PATH / f"isfs_m2hats_qc_geo_hr_{ts.strftime('%Y%m%d_%H')}.nc"
raw_ds   = fix_time_and_t1(xr.open_dataset(raw_file, decode_timedelta=False))
raw_slice = raw_ds.sel(time=CHECK_TIME, method="nearest")

# Sample dim for the group
sample_dim = {"60hz": "sample", "30hz": "sample_30", "50hz": "sample_50"}
sdim = next(v for k, v in sample_dim.items() if k in CHECK_GROUP)

raw_site = {
    v: raw_slice[v].values
    for v in raw_ds.data_vars
    if v.endswith(f"_{CHECK_SITE}") and sdim in raw_ds[v].dims
}

# ── 3. Compare ────────────────────────────────────────────────────────────────
# reverse map: output name → ISFS short name (e.g. "wind_u" → "u")
rev_map = {meta[0]: short for short, meta in VAR_METADATA.items()}

print(f"site={CHECK_SITE}  time={CHECK_TIME}")
print(f"zarr group : {CHECK_GROUP}")
print(f"raw file   : {raw_file.name}\n")
print(f"{'zarr var':25s}  {'raw var':25s}  {'match':>6s}  {'max_diff':>12s}")
print("─" * 75)

for zarr_var in sorted(zarr_slice.data_vars):
    isfs_short = rev_map.get(zarr_var)
    if isfs_short is None:
        continue
    candidates = [v for v in raw_site if v.startswith(f"{isfs_short}_")]
    if not candidates:
        print(f"{zarr_var:25s}  {'(no match in raw)':25s}")
        continue
    raw_var = candidates[0]

    z = zarr_slice[zarr_var].values.astype("float64")
    r = raw_site[raw_var].astype("float64")
    #print(z, r)

    both_nan = np.isnan(z) & np.isnan(r)
    match    = bool(np.all(both_nan | (np.abs(z - r) < 1e-5)))
    max_diff = float(np.nanmax(np.abs(z - r))) if not np.all(both_nan) else float("nan")

    print(f"{zarr_var:25s}  {raw_var:25s}  {'✓' if match else '✗':>6s}  {max_diff:>12.2e}")

raw_ds.close()

site=t0p  time=2023-09-21T09:12:02.500000000
zarr group : /array/sonic_60hz
raw file   : isfs_m2hats_qc_geo_hr_20230921_09.nc

zarr var                   raw var                     match      max_diff
───────────────────────────────────────────────────────────────────────────
sonic_diag_bits            (no match in raw)        
sonic_qc_flag              ldiag_4m_t0p                    ✓      0.00e+00
sonic_temperature          tc_4m_t0p                       ✓      0.00e+00
wind_u                     u_4m_t0p                        ✓      0.00e+00
wind_v                     v_4m_t0p                        ✓      0.00e+00
wind_w                     w_4m_t0p                        ✓      0.00e+00


In [38]:
cluster.close()
client.close()